In [3]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [4]:
FEATURE_PATH = Path("door_cycle_features.csv")

print("Feature file exists:", FEATURE_PATH.exists())

cycle_features = pd.read_csv(FEATURE_PATH)

print("Feature table shape:", cycle_features.shape)

display(cycle_features.head())

Feature file exists: True
Feature table shape: (110, 143)


,Motor current(mA)_mean,Motor current(mA)_std,Motor current(mA)_min,Motor current(mA)_max,Motor current(mA)_range,Motor Voltage(10mV)_mean,Motor Voltage(10mV)_std,Motor Voltage(10mV)_min,Motor Voltage(10mV)_max,Motor Voltage(10mV)_range,...,position_zero_change_fraction,current_abs_integral,voltage_abs_integral,force_abs_integral,duration_seconds,n_rows,is_close,segment_id,operation,status
0,430.698925,523.670601,0.0,2208.0,2208.0,4146.236559,1416.459249,400.0,6200.0,5800.0,...,0.070270,80110.0,771200.0,165925.0,3.70,186,1,train_seg_001,Close,Normal
1,636.265734,727.547100,6.0,2493.0,2487.0,5521.678322,3184.691088,300.0,9100.0,8800.0,...,0.147887,90986.0,789600.0,168558.0,2.84,143,0,train_seg_002,Open,Normal
2,862.452555,777.944043,9.0,2493.0,2484.0,6197.080292,3478.273370,400.0,10400.0,10000.0,...,0.147059,118156.0,849000.0,163363.0,2.72,137,0,train_seg_003,Open,Abnormal resistance
3,576.577540,457.693062,112.0,1993.0,1881.0,4524.598930,1461.716164,300.0,6800.0,6500.0,...,0.075269,107820.0,846100.0,164243.0,3.72,187,1,train_seg_004,Close,Abnormal resistance
4,484.548387,446.145295,85.0,1993.0,1908.0,4339.247312,1325.237846,300.0,6400.0,6100.0,...,0.070270,90126.0,807100.0,163964.0,3.70,186,1,train_seg_005,Close,Abnormal resistance


In [5]:
# Separate numerical features from metadata and encode the classification target.

metadata_columns = ["segment_id", "operation", "status"]
numeric_features = [column for column in cycle_features.columns if column not in metadata_columns]

X = cycle_features[numeric_features].copy()
y = cycle_features["status"].map({"Normal": 0, "Abnormal resistance": 1})

print("Samples:", len(X))
print("Features:", X.shape[1])
print("Normal:", (y == 0).sum())
print("Abnormal resistance:", (y == 1).sum())

Samples: 110
Features: 140
Normal: 80
Abnormal resistance: 30


In [6]:
# Reconstruct the feature groups used during the earlier robustness experiments.


def get_feature_group(feature):
    if "_phase" in feature:
        return "Phase"
    if "_diff_" in feature or feature.startswith("position_abs_diff") or feature.startswith("position_zero_change"):
        return "Dynamic"
    if "integral" in feature:
        return "Integral"
    if feature in ["duration_seconds", "n_rows"]:
        return "Timing"
    if feature == "is_close":
        return "Operation"
    return "Whole-cycle"


feature_groups = pd.DataFrame({"feature": X.columns, "group": [get_feature_group(feature) for feature in X.columns]})

display(feature_groups["group"].value_counts())

group
Phase          99
Whole-cycle    19
Dynamic        16
Integral        3
Timing          2
Operation       1
Name: count, dtype: int64

In [7]:
# Create the three candidate representations we want to compare.

whole_cycle_features = feature_groups.loc[feature_groups["group"] == "Whole-cycle", "feature"].tolist()

candidate_sets = {"Whole-cycle": whole_cycle_features, "All features": X.columns.tolist()}

print("Whole-cycle features:", len(whole_cycle_features))
print("All features:", len(X.columns))

Whole-cycle features: 19
All features: 140


In [8]:
# Identify a 40-feature candidate using SelectKBest for comparison with the compact representations.

selector_40 = SelectKBest(score_func=f_classif, k=40)
selector_40.fit(X, y)

selected_40 = X.columns[selector_40.get_support()].tolist()

candidate_sets["40 selected"] = selected_40

print("40 selected features:", len(selected_40))

40 selected features: 40


In [9]:
# Compare the candidate feature representations using stratified 5-fold cross-validation.

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

feature_set_results = []

for name, feature_list in candidate_sets.items():
    X_candidate = X[feature_list]
    predictions = np.zeros(len(y), dtype=int)

    for train_idx, val_idx in cv.split(X_candidate, y):
        model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

        model.fit(X_candidate.iloc[train_idx], y.iloc[train_idx])

        predictions[val_idx] = model.predict(X_candidate.iloc[val_idx])

    feature_set_results.append(
        {
            "feature_set": name,
            "feature_count": len(feature_list),
            "accuracy": accuracy_score(y, predictions),
            "macro_f1": f1_score(y, predictions, average="macro"),
            "abnormal_recall": recall_score(y, predictions, zero_division=0),
            "abnormal_f1": f1_score(y, predictions, zero_division=0),
        }
    )

feature_set_results = pd.DataFrame(feature_set_results)

display(feature_set_results.sort_values(["accuracy", "feature_count"], ascending=[False, True]))

,feature_set,feature_count,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,Whole-cycle,19,1.000000,1.00000,1.000000,1.000000
1,All features,140,1.000000,1.00000,1.000000,1.000000
2,40 selected,40,0.990909,0.98842,0.966667,0.983051


In [10]:
# Evaluate each candidate feature set by training on earlier cycles and validating on later cycles.

forward_splits = [0.4, 0.5, 0.6, 0.7, 0.8]

forward_feature_results = []

for feature_set_name, feature_list in candidate_sets.items():
    X_candidate = X[feature_list]

    for fraction in forward_splits:
        split = int(len(y) * fraction)

        train_idx = np.arange(0, split)
        val_idx = np.arange(split, len(y))

        if y.iloc[train_idx].nunique() < 2 or y.iloc[val_idx].nunique() < 2:
            continue

        model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

        model.fit(X_candidate.iloc[train_idx], y.iloc[train_idx])

        predictions = model.predict(X_candidate.iloc[val_idx])

        forward_feature_results.append(
            {
                "feature_set": feature_set_name,
                "train_fraction": fraction,
                "accuracy": accuracy_score(y.iloc[val_idx], predictions),
                "macro_f1": f1_score(y.iloc[val_idx], predictions, average="macro"),
                "abnormal_recall": recall_score(y.iloc[val_idx], predictions, zero_division=0),
                "abnormal_f1": f1_score(y.iloc[val_idx], predictions, zero_division=0),
            }
        )

forward_feature_results = pd.DataFrame(forward_feature_results)

display(forward_feature_results)

,feature_set,train_fraction,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,Whole-cycle,0.4,0.984848,0.981803,0.95,0.974359
1,Whole-cycle,0.5,1.000000,1.000000,1.00,1.000000
2,Whole-cycle,0.6,1.000000,1.000000,1.00,1.000000
3,Whole-cycle,0.7,1.000000,1.000000,1.00,1.000000
4,Whole-cycle,0.8,1.000000,1.000000,1.00,1.000000
5,All features,0.4,1.000000,1.000000,1.00,1.000000
6,All features,0.5,1.000000,1.000000,1.00,1.000000
7,All features,0.6,1.000000,1.000000,1.00,1.000000
8,All features,0.7,1.000000,1.000000,1.00,1.000000
9,All features,0.8,1.000000,1.000000,1.00,1.000000


In [11]:
# Summarise the forward-time performance of each feature representation.

display(forward_feature_results.groupby("feature_set")[["accuracy", "macro_f1", "abnormal_recall", "abnormal_f1"]].agg(["mean", "std", "min"]))

accuracy                      macro_f1                      \
                 mean       std       min      mean       std       min   
feature_set                                                               
40 selected   1.00000  0.000000  1.000000  1.000000  0.000000  1.000000   
All features  1.00000  0.000000  1.000000  1.000000  0.000000  1.000000   
Whole-cycle   0.99697  0.006776  0.984848  0.996361  0.008138  0.981803   

             abnormal_recall                 abnormal_f1                      
                        mean       std   min        mean       std       min  
feature_set                                                                   
40 selected             1.00  0.000000  1.00    1.000000  0.000000  1.000000  
All features            1.00  0.000000  1.00    1.000000  0.000000  1.000000  
Whole-cycle             0.99  0.022361  0.95    0.994872  0.011467  0.974359

In [12]:
# Display the main comparison used to select the final feature representation.

summary = feature_set_results.copy()

forward_summary = (
    forward_feature_results.groupby("feature_set")["accuracy"]
    .agg(["mean", "min"])
    .rename(columns={"mean": "forward_mean_accuracy", "min": "forward_min_accuracy"})
)

summary = summary.merge(forward_summary, left_on="feature_set", right_index=True)

display(summary.sort_values(["accuracy", "forward_min_accuracy", "feature_count"], ascending=[False, False, True]))

,feature_set,feature_count,accuracy,macro_f1,abnormal_recall,abnormal_f1,forward_mean_accuracy,forward_min_accuracy
1,All features,140,1.000000,1.00000,1.000000,1.000000,1.00000,1.000000
0,Whole-cycle,19,1.000000,1.00000,1.000000,1.000000,0.99697,0.984848
2,40 selected,40,0.990909,0.98842,0.966667,0.983051,1.00000,1.000000


In [13]:
# Set the final feature representation based on the validation results.

FINAL_FEATURE_SET = "Whole-cycle"

FINAL_FEATURES = candidate_sets[FINAL_FEATURE_SET]

print("Final feature set:", FINAL_FEATURE_SET)
print("Number of features:", len(FINAL_FEATURES))

for feature in FINAL_FEATURES:
    print(feature)

Final feature set: Whole-cycle
Number of features: 19
Motor current(mA)_mean
Motor current(mA)_std
Motor current(mA)_min
Motor current(mA)_max
Motor current(mA)_range
Motor Voltage(10mV)_mean
Motor Voltage(10mV)_std
Motor Voltage(10mV)_min
Motor Voltage(10mV)_max
Motor Voltage(10mV)_range
Motor electrodynamic force_mean
Motor electrodynamic force_std
Motor electrodynamic force_max
Motor electrodynamic force_range
Door leaf position_mean
Door leaf position_std
Door leaf position_min
Door leaf position_max
Door leaf position_range


## Train Final Model

In [14]:
# Train the final Logistic Regression pipeline on all 110 labelled cycles.

X_final = X[FINAL_FEATURES]

final_model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

final_model.fit(X_final, y)

print("Final model trained.")
print("Training samples:", len(X_final))
print("Features:", X_final.shape[1])

Final model trained.
Training samples: 110
Features: 19


In [15]:
# Check that the final model can reproduce the training labels before saving it.

training_predictions = final_model.predict(X_final)

print("Training accuracy:", accuracy_score(y, training_predictions))

print("Training macro F1:", f1_score(y, training_predictions, average="macro"))

print("Training abnormal recall:", recall_score(y, training_predictions, zero_division=0))

Training accuracy: 1.0
Training macro F1: 1.0
Training abnormal recall: 1.0


In [16]:
# Save the complete preprocessing + classifier pipeline for Notebook 7 inference.

MODEL_PATH = Path("door_final_model.joblib")

joblib.dump(final_model, MODEL_PATH)

print("Saved:", MODEL_PATH.resolve())

Saved: /Users/yeo/Documents/Door/NOTEBOOKS/door_final_model.joblib


In [17]:
# Save the exact feature list required by the final model.

FEATURE_LIST_PATH = Path("door_final_features.csv")

pd.DataFrame({"feature": FINAL_FEATURES}).to_csv(FEATURE_LIST_PATH, index=False)

print("Saved:", FEATURE_LIST_PATH.resolve())

Saved: /Users/yeo/Documents/Door/NOTEBOOKS/door_final_features.csv


In [18]:
# Save basic metadata describing the final Door model.

metadata = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "feature_set": FINAL_FEATURE_SET,
            "feature_count": len(FINAL_FEATURES),
            "training_samples": len(X_final),
            "normal_samples": int((y == 0).sum()),
            "abnormal_samples": int((y == 1).sum()),
        }
    ]
)

metadata.to_csv("door_final_model_metadata.csv", index=False)

display(metadata)

,model,feature_set,feature_count,training_samples,normal_samples,abnormal_samples
0,Logistic Regression,Whole-cycle,19,110,80,30
